# Supplementary Figure 2 — GTEx

Everything in this figure comes from the GTEx arm of the pipeline. It was split out
of Supplementary Figure 1, which had reached the full A4 page with eight panels drawn
from two different data sources; separating them by source means each supplement is
fed by a single rule chain and can grow independently.

- **A** — how orthogonal the GO:BP training prior is to the gene sets the models are
  evaluated against, which is what stops the marker-recovery results from being
  circular.
- **B** — robustness of RNA-Seq tissue clustering to gene subsampling.
- **C** — method-level tissue-clustering benchmark.
- **D** — subtissue-level B-matrix concordance, confirmed against the Z matrix.

💡 **Environment:** `clamp-analyses`

In [ ]:
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
  library(patchwork)
  library(cowplot)
  library(yaml)
  library(here)
  library(grid)
  library(ragg)
  library(svglite)
})


In [ ]:
FONT_FAMILY <- "Helvetica"

FS_TAG          <- 9
FS_TITLE        <- 8
FS_SUBTITLE     <- 6.5
FS_AXIS_TITLE   <- 7
FS_AXIS_TEXT    <- 6.5
FS_LEGEND       <- 5.5
FS_LEGEND_TITLE <- 6
FS_STAT         <- 6
FS_HEAT_LABEL   <- 5.5
FS_HEAT_VALUE   <- 5.5
FS_CELL_VALUE   <- 5
FS_DENSE_VALUE  <- 4
FS_A_MEAN       <- 4
FS_HEAT_LABEL_H <- 5

LINE_W <- 0.3
GRID_W <- 0.2

pt_mm <- function(pt) pt / .pt

theme_nature_methods <- function(base_size = FS_AXIS_TEXT,
                                 grid = c("none", "y", "x", "both")) {
  grid <- match.arg(grid)
  th <- theme_classic(base_size = base_size, base_family = FONT_FAMILY) %+replace%
    theme(
      plot.title        = element_text(size = FS_TITLE, face = "bold", hjust = 0.5,
                                       margin = margin(b = 1)),
      plot.subtitle     = element_text(size = FS_SUBTITLE, face = "plain", hjust = 0.5,
                                       lineheight = 0.95, margin = margin(b = 1)),
      plot.tag          = element_text(size = FS_TAG, face = "bold", family = FONT_FAMILY),
      plot.tag.position = "topleft",
      plot.tag.location = "margin",
      axis.line         = element_line(linewidth = LINE_W, colour = "black"),
      axis.ticks        = element_line(linewidth = LINE_W, colour = "black"),
      axis.ticks.length = unit(0.9, "pt"),
      axis.text         = element_text(size = base_size, colour = "black"),
      axis.text.x       = element_text(margin = margin(t = 0.8)),
      axis.text.y       = element_text(hjust = 1, margin = margin(r = 0.8)),
      axis.title        = element_text(size = FS_AXIS_TITLE, colour = "black"),
      axis.title.x      = element_text(margin = margin(t = 1)),
      axis.title.y      = element_text(angle = 90, margin = margin(r = 1)),
      legend.text       = element_text(size = FS_LEGEND),
      legend.title      = element_text(size = FS_LEGEND_TITLE),
      legend.key.size   = unit(2, "mm"),
      legend.key        = element_blank(),
      legend.background = element_blank(),
      legend.margin     = margin(0, 0, 0, 0),
      legend.box.margin = margin(0, 0, 0, 0),
      strip.text        = element_text(size = FS_TITLE, face = "bold", margin = margin(b = 1)),
      strip.background  = element_blank(),
      panel.background  = element_blank(),
      panel.grid        = element_blank(),
      plot.background   = element_blank(),
      plot.margin       = margin(1, 1, 1, 1, "mm")
    )
  if (grid %in% c("y", "both"))
    th <- th + theme(panel.grid.major.y = element_line(colour = "grey88", linewidth = GRID_W))
  if (grid %in% c("x", "both"))
    th <- th + theme(panel.grid.major.x = element_line(colour = "grey88", linewidth = GRID_W))
  th
}

theme_nature_heatmap <- function(x_angle = 45, label_size = FS_HEAT_LABEL) {
  theme_nature_methods() %+replace%
    theme(
      axis.line   = element_blank(),
      axis.ticks  = element_blank(),
      axis.text.x = element_text(size = label_size, angle = x_angle,
                                 hjust = if (x_angle == 0) 0.5 else 1,
                                 vjust = if (x_angle == 90) 0.5 else 1,
                                 colour = "black",
                                 margin = margin(t = 0.5), lineheight = 0.9),
      axis.text.y = element_text(size = label_size, hjust = 1, colour = "black",
                                 margin = margin(r = 0.5), lineheight = 0.9),
      plot.margin = margin(1, 1, 1, 1, "mm")
    )
}

add_tag <- function(p, tag) {
  p + labs(tag = tag) +
    theme(plot.tag          = element_text(size = FS_TAG, face = "bold",
                                           family = FONT_FAMILY),
          plot.tag.position = "topleft",
          plot.tag.location = "margin")
}

add_overlay_tag <- function(p, tag) {
  tagged <- ggdraw(p) +
    draw_label(tag, x = 0, y = 1, hjust = 0, vjust = 1,
               size = FS_TAG, fontface = "bold", fontfamily = FONT_FAMILY)
  wrap_elements(full = tagged)
}

DATASET_LABELS <- c(
  Brain_Mathys2023 = "Brain Mathys",
  Brain_Xiong2023  = "Brain Xiong",
  Heart_Datar2026  = "Heart Datar",
  PBMC_1k1k        = "PBMC 1k1k",
  PBMC_Perez2022   = "PBMC Perez",
  Lung_Sikkema2023 = "Lung Sikkema"
)

TISSUE_MAP <- c(
  Brain_Mathys2023 = "Brain", Brain_Xiong2023 = "Brain",
  Heart_Datar2026  = "Heart",
  Lung_Sikkema2023 = "Lung",
  PBMC_1k1k        = "PBMC", PBMC_Perez2022 = "PBMC"
)

DATASETS_ROW1 <- c("Heart_Datar2026", "PBMC_1k1k", "Lung_Sikkema2023")
DATASETS_ROW2 <- c("Brain_Mathys2023", "Brain_Xiong2023", "PBMC_Perez2022")


TISSUE_ORDER <- c("Brain", "Heart", "Lung", "PBMC")

abbreviate_ct <- function(x) {
  x <- gsub("Oligodendrocyte [Pp]rogenitor [Cc]ells?", "OPC", x)
  x <- gsub("Oligodendrocyte [Pp]recursor [Cc]ells?", "OPC", x)
  x
}

CT_SHORT <- c(
  "Oligodendrocyte Precursor Cells" = "OPC",
  "Plasmacytoid Dendritic Cells"    = "pDC",
  "LymphaticEndothelial"            = "Lymphatic EC",
  "Alveolar epithelium"             = "Alveolar ep.",
  "Airway epithelium"               = "Airway ep.",
  "Excitatory neurons"              = "Excitatory",
  "Inhibitory neurons"              = "Inhibitory",
  "Endothelial cells"               = "Endothelial",
  "Fibroblast lineage"              = "Fibroblast",
  "Submucosal Gland"                = "Submucosal",
  "Oligodendrocytes"                = "Oligodendro.",
  "CD14+ Monocytes"                 = "CD14+ Mono",
  "CD16+ Monocytes"                 = "CD16+ Mono",
  "Dendritic cells"                 = "Dendritic",
  "Plasma B cells"                  = "Plasma B",
  "Myeloid cells"                   = "Myeloid",
  "Malignant cells"                 = "Malignant",
  "Vascular cells"                  = "Vascular",
  "Cancer cells"                    = "Cancer",
  "Mast cells"                      = "Mast",
  "Blood vessels"                   = "Blood vessel"
)
short_ct <- function(x) {
  x <- abbreviate_ct(x)
  ifelse(x %in% names(CT_SHORT), CT_SHORT[x], x)
}

wrap_label <- function(x, width = 14) {
  vapply(x, function(s) paste(strwrap(s, width = width), collapse = "\n"),
         character(1), USE.NAMES = FALSE)
}

fmt_corr_cell <- function(x) sub("0.", ".", sprintf("%.2f", x), fixed = TRUE)

shorten_gtex <- function(x) {
  x <- gsub("Adipose - Subcutaneous",                    "Adipose - Subcut.", x)
  x <- gsub("Adipose - Visceral \\(Omentum\\)",             "Adipose - Visceral", x)
  x <- gsub("Artery - ",                                 "Artery - ", x)
  x <- gsub("Brain - Amygdala",                          "Brain - Amygdala", x)
  x <- gsub("Brain - Anterior cingulate cortex \\(BA24\\)", "Brain - ACC (BA24)", x)
  x <- gsub("Brain - Caudate \\(basal ganglia\\)",         "Brain - Caudate", x)
  x <- gsub("Brain - Cerebellar Hemisphere",             "Brain - Cereb. hem.", x)
  x <- gsub("Brain - Frontal Cortex \\(BA9\\)",            "Brain - FC (BA9)", x)
  x <- gsub("Brain - Nucleus accumbens \\(basal ganglia\\)", "Brain - NAc", x)
  x <- gsub("Brain - Putamen \\(basal ganglia\\)",         "Brain - Putamen", x)
  x <- gsub("Brain - Spinal cord \\(cervical c-1\\)",      "Brain - Spinal cord", x)
  x <- gsub("Brain - Substantia nigra",                  "Brain - Subst. nigra", x)
  x <- gsub("Breast - Mammary Tissue",                   "Breast - Mammary", x)
  x <- gsub("Cells - Cultured fibroblasts",              "Cells - Fibroblasts", x)
  x <- gsub("Cells - EBV-transformed lymphocytes",       "Cells - EBV lymph.", x)
  x <- gsub("Esophagus - Gastroesophageal Junction",     "Esoph. - GE junction", x)
  x <- gsub("Esophagus - Mucosa",                        "Esoph. - Mucosa", x)
  x <- gsub("Esophagus - Muscularis",                    "Esoph. - Muscularis", x)
  x <- gsub("Heart - Atrial Appendage",                  "Heart - Atrial app.", x)
  x <- gsub("Minor Salivary Gland",                      "Minor saliv. gland", x)
  x <- gsub("Skin - Not Sun Exposed \\(Suprapubic\\)",     "Skin - Not sun exp.", x)
  x <- gsub("Skin - Sun Exposed \\(Lower leg\\)",          "Skin - Sun exp.", x)
  x <- gsub("Small Intestine - Terminal Ileum",          "Sm. intestine - Ileum", x)
  trimws(x)
}

fmt_q_compact <- function(q) {
  if (is.na(q)) return('"n.a."')
  if (q >= 0.001) return(sprintf('"%.3f"', q))
  e_str    <- formatC(q, format = "e", digits = 1)
  parts    <- strsplit(e_str, "e")[[1]]
  sprintf('%s%%*%%10^{%d}', trimws(parts[1]), as.integer(parts[2]))
}

assign_bracket_tiers <- function(comp_df, xpos) {
  comp_ord <- copy(as.data.table(comp_df))
  comp_ord[, x1 := xpos[a]]
  comp_ord[, x2 := xpos[b]]
  comp_ord[, left  := pmin(x1, x2)]
  comp_ord[, right := pmax(x1, x2)]
  comp_ord[, span  := right - left]
  setorder(comp_ord, span, q)

  levels_used <- list()
  comp_ord[, tier := 0L]
  for (i in seq_len(nrow(comp_ord))) {
    left  <- comp_ord$left[i]; right <- comp_ord$right[i]; tier <- 1L
    repeat {
      current  <- if (tier <= length(levels_used)) levels_used[[tier]] else NULL
      overlaps <- !is.null(current) && any(vapply(current, function(iv) {
        !(right < iv[1] || left > iv[2])
      }, logical(1)))
      if (!overlaps) break
      tier <- tier + 1L
    }
    comp_ord$tier[i] <- tier
    prior <- if (tier <= length(levels_used)) levels_used[[tier]] else list()
    levels_used[[tier]] <- c(prior, list(c(left, right)))
  }
  comp_ord
}

add_brackets <- function(p, comp_ord, y_base, y_step, h, size = pt_mm(FS_STAT)) {
  for (i in seq_len(nrow(comp_ord))) {
    r <- comp_ord[i, ]
    y <- y_base + (r$tier - 1) * y_step
    p <- p +
      annotate("segment", x = r$left,  xend = r$left,  y = y,     yend = y + h, linewidth = 0.2) +
      annotate("segment", x = r$left,  xend = r$right, y = y + h, yend = y + h, linewidth = 0.2) +
      annotate("segment", x = r$right, xend = r$right, y = y,     yend = y + h, linewidth = 0.2) +
      annotate("text", x = (r$left + r$right) / 2, y = y + h + 0.005,
               label = fmt_q_compact(r$q), size = size, vjust = 0,
               parse = TRUE, family = FONT_FAMILY)
  }
  p
}

cfg <- yaml::read_yaml(here("config.yaml"))
MODEL_COLORS_RAW <- unlist(cfg$MODEL_COLORS)
names(MODEL_COLORS_RAW)[names(MODEL_COLORS_RAW) == "GenomicSuperSignature"] <- "GSSig"

ct_labels_df <- read.csv(here("data", "pseudobulk", "cell_type_labels.csv"), stringsAsFactors = FALSE)
CT_LABELS <- setNames(ct_labels_df$label, ct_labels_df$cell_type)
ct_label <- function(x) ifelse(x %in% names(CT_LABELS), CT_LABELS[x], x)

GREEN_SCALE      <- unlist(cfg$GREEN_SCALE)
DIVERGING_COLORS <- unlist(cfg$DIVERGING_COLORS)
DIVERGING_VALUES <- as.numeric(cfg$DIVERGING_VALUES)

TILE_TEXT <- c(`TRUE` = "white", `FALSE` = "black")


## Panel A: GO:BP prior vs the evaluation gene sets

For every evaluation term, the maximum Jaccard index against any GO:BP training term.
Complete redundancy with a training term would sit at 1.0. The size- and
coverage-matched null is computed in `07_geneset_orthogonality.ipynb` and kept in the
per-term CSV; plotting it here would imply the comparison is close enough to need a
null, which the observed distribution already answers.

In [ ]:
ortho <- fread(snakemake@input[["orthogonality"]])
ortho_summary <- fread(snakemake@input[["orthogonality_summary"]])

ORTHO_LEVELS <- c("GTEx Tissues", "CellMarker Human", "Allen Brain Atlas")
ortho[, collection := factor(collection, levels = ORTHO_LEVELS)]
stopifnot(!any(is.na(ortho$collection)))

ortho_means <- ortho[, .(m = mean(max_jaccard), n = .N), by = collection]
ortho_means[, lab := sprintf("mean %.3f\n(n = %d)", m, n)]

panel_A <- ggplot(ortho, aes(x = max_jaccard)) +
  geom_density(fill = "#2166AC", colour = "#2166AC", alpha = 0.35, adjust = 1.2) +
  geom_vline(data = ortho_means, aes(xintercept = m),
             linetype = "dashed", linewidth = 0.3, colour = "black") +
  geom_text(data = ortho_means, aes(x = m, y = Inf, label = lab),
            hjust = -0.12, vjust = 1.35, size = pt_mm(FS_STAT), colour = "black",
            lineheight = 0.95) +
  facet_wrap(~collection, scales = "free_y", ncol = 1) +
  coord_cartesian(xlim = c(0, 0.45)) +
  labs(x = "Max Jaccard against any GO:BP training term", y = "Density") +
  theme_nature_methods() +
  theme(strip.background = element_blank(),
        strip.text = element_text(face = "bold", size = FS_TITLE),
        panel.grid = element_blank())
panel_A <- add_overlay_tag(panel_A, "A")

cat(sprintf("%d evaluation terms; %d above Jaccard 0.5, %d above overlap coefficient 0.8\n",
            nrow(ortho), sum(ortho$max_jaccard > 0.5),
            sum(ortho$max_overlap_coef > 0.8)))
print(as.data.frame(ortho_summary[, .(collection, n_terms, mean_max_jaccard,
                                      p90_max_jaccard, max_max_jaccard,
                                      `n_jaccard_gt_0.5`)]), row.names = FALSE)

## Panel B: RNA-Seq robustness to gene subsampling (ARI)

In [ ]:
gene_frac_ari         <- fread(snakemake@input[["gene_fraction_ari_data"]])
gene_frac_comparisons <- fread(snakemake@input[["gene_fraction_ari_comparisons"]])
stopifnot(all(c("fraction", "ari") %in% names(gene_frac_ari)))
stopifnot(all(c("a", "b", "q") %in% names(gene_frac_comparisons)))

FRACTION_LEVELS <- c("100%", "75%", "50%", "25%", "10%", "5%", "1%")
FRACTION_TICKS  <- sub("%$", "", FRACTION_LEVELS)
gene_frac_ari[, fraction := factor(fraction, levels = FRACTION_LEVELS)]
stopifnot(!anyNA(gene_frac_ari$fraction))

FRACTION_COLORS <- unlist(cfg$RNASEQ_FRACTION_COLORS)[FRACTION_LEVELS]

mean_df_genefrac <- gene_frac_ari[, .(mean_ari = mean(ari, na.rm = TRUE),
                                      max_ari  = max(ari, na.rm = TRUE)), by = fraction]

GENEFRAC_SHOWN <- c("75%", "50%", "10%", "1%")
comp_genefrac  <- gene_frac_comparisons[a == "100%" & b %in% GENEFRAC_SHOWN]
stopifnot(nrow(comp_genefrac) == length(GENEFRAC_SHOWN))

xpos_genefrac  <- setNames(seq_along(FRACTION_LEVELS), FRACTION_LEVELS)
comp_genefrac  <- assign_bracket_tiers(comp_genefrac, xpos_genefrac)

y_base_genefrac <- max(mean_df_genefrac$max_ari) + 0.04
y_step_genefrac <- 0.090
h_genefrac      <- 0.012
y_max_genefrac  <- y_base_genefrac + (max(comp_genefrac$tier) - 1) * y_step_genefrac +
                   h_genefrac + 0.085

panel_B <- ggplot(gene_frac_ari, aes(fraction, ari, fill = fraction)) +
  geom_boxplot(width = 0.55, outlier.shape = NA, colour = "black",
               linewidth = 0.25, alpha = 0.85) +
  geom_jitter(width = 0.10, size = 0.3, shape = 21, fill = "white",
              colour = "#333333", stroke = 0.12, alpha = 0.75) +
  geom_point(data = mean_df_genefrac, aes(x = fraction, y = mean_ari), shape = 23,
             size = 1.1, fill = "white", colour = "black", stroke = 0.3,
             inherit.aes = FALSE) +
  scale_fill_manual(values = FRACTION_COLORS, na.value = "grey70") +
  scale_x_discrete(labels = setNames(FRACTION_TICKS, FRACTION_LEVELS)) +
  scale_y_continuous(breaks = seq(0, 1, 0.25),
                     labels = c("0", "0.25", "0.5", "0.75", "1.0"),
                     expand = expansion(mult = c(0.02, 0.02))) +
  coord_cartesian(ylim = c(0, y_max_genefrac), clip = "on") +
  labs(x = "Genes used for RNA-Seq (%)", y = "Adjusted Rand Index (ARI)") +
  theme_nature_methods(grid = "none") +
  theme(legend.position = "none",
        axis.text.x = element_text(angle = 60, hjust = 1, vjust = 1, size = FS_LEGEND_TITLE),
        axis.title.y = element_text(angle = 90, margin = margin(r = 0.3)),
        plot.margin = margin(1, 1, 1, 0.3, "mm"))

panel_B <- add_brackets(panel_B, comp_genefrac,
                        y_base = y_base_genefrac, y_step = y_step_genefrac, h = h_genefrac,
                        size = pt_mm(FS_HEAT_LABEL_H))
panel_B <- add_overlay_tag(panel_B, "B")

options(repr.plot.width = 3.6, repr.plot.height = 2.2)
print(panel_B)

## Panel C: GTEx method-level tissue-clustering benchmark (ARI)

In [ ]:
ari_data        <- fread(snakemake@input[["ari_data"]])
ari_comparisons <- fread(snakemake@input[["ari_comparisons"]])
stopifnot(all(c("method", "ari") %in% names(ari_data)))
stopifnot(all(c("a", "b", "q") %in% names(ari_comparisons)))

ari_data[, method := fifelse(method == "GenomicSuperSignature", "GSSig", method)]
ari_comparisons[, a := fifelse(a == "GenomicSuperSignature", "GSSig", a)]
ari_comparisons[, b := fifelse(b == "GenomicSuperSignature", "GSSig", b)]

ari_method_means <- ari_data[, .(m = mean(ari, na.rm = TRUE)), by = method]
G_PINNED_ORDER <- c("CLAMPfull", "PLIER", "CLAMPbase", "NMF")
G_PINNED_ORDER <- G_PINNED_ORDER[G_PINNED_ORDER %in% ari_method_means$method]
method_order_ari <- c(
  G_PINNED_ORDER,
  ari_method_means[!method %in% G_PINNED_ORDER][order(-m), as.character(method)]
)
ari_data[, method := factor(method, levels = method_order_ari)]

METHOD_COLORS_ARI <- MODEL_COLORS_RAW[method_order_ari]
METHOD_COLORS_ARI[is.na(METHOD_COLORS_ARI)] <- "grey70"
names(METHOD_COLORS_ARI) <- method_order_ari

mean_df_ari <- ari_data[, .(mean_ari = mean(ari, na.rm = TRUE),
                            max_ari  = max(ari, na.rm = TRUE)), by = method]
mean_df_ari[, method := factor(method, levels = method_order_ari)]

comp_ari <- copy(ari_comparisons)
xpos_ari <- setNames(seq_along(method_order_ari), method_order_ari)
comp_ari <- assign_bracket_tiers(comp_ari, xpos_ari)

y_base_ari <- max(mean_df_ari$max_ari) + 0.05
y_step_ari <- 0.080
h_ari      <- 0.015
y_max_ari  <- y_base_ari + (max(comp_ari$tier) - 1) * y_step_ari + h_ari + 0.085

panel_C <- ggplot(ari_data, aes(method, ari, fill = method)) +
  geom_boxplot(width = 0.55, outlier.shape = NA, colour = "black",
               linewidth = 0.25, alpha = 0.85) +
  geom_jitter(width = 0.10, size = 0.3, shape = 21, fill = "white",
              colour = "#333333", stroke = 0.12, alpha = 0.75) +
  geom_point(data = mean_df_ari, aes(x = method, y = mean_ari), shape = 23,
             size = 1.1, fill = "white", colour = "black", stroke = 0.3,
             inherit.aes = FALSE) +
  scale_fill_manual(values = METHOD_COLORS_ARI, na.value = "grey70") +
  scale_y_continuous(breaks = seq(0, 1, 0.25),
                     labels = c("0", "0.25", "0.5", "0.75", "1.0"),
                     expand = expansion(mult = c(0.02, 0.02))) +
  coord_cartesian(ylim = c(0, y_max_ari), clip = "on") +
  labs(x = NULL, y = "Adjusted Rand Index (ARI)") +
  theme_nature_methods(grid = "none") +
  theme(legend.position = "none",
        axis.text.x = element_text(angle = 35, hjust = 1, vjust = 1, size = FS_AXIS_TEXT),
        axis.title.y = element_text(angle = 90, margin = margin(r = 0.3)),
        plot.margin = margin(1, 1, 1, 0.3, "mm"))

panel_C <- add_brackets(panel_C, comp_ari,
                        y_base = y_base_ari, y_step = y_step_ari, h = h_ari,
                        size = pt_mm(FS_HEAT_LABEL_H))
panel_C <- add_overlay_tag(panel_C, "C")

options(repr.plot.width = 3.6, repr.plot.height = 2.2)
print(panel_C)

## Panel D: subtissue-level B-matrix concordance, with Z-matrix confirmation

In [ ]:
tissue_subtissue_heatmap <- fread(snakemake@input[["tissue_subtissue_heatmap"]])
z_matrix_subtissue       <- fread(snakemake@input[["z_matrix_subtissue"]])
stopifnot(all(c("Tissue", "Predicted_Tissue", "Pct") %in% names(tissue_subtissue_heatmap)))

hm <- copy(tissue_subtissue_heatmap)

hm_order  <- sort(unique(hm$Tissue))
hm_short  <- shorten_gtex(hm_order)
N_SUBTISSUES <- length(hm_order)
stopifnot(!anyDuplicated(hm_short))

hm[, row_lab := factor(shorten_gtex(Tissue), levels = rev(hm_short))]
hm[, col_lab := factor(shorten_gtex(Predicted_Tissue), levels = hm_short)]

SUBTISSUE_LABEL_MIN <- 20
hm[, is_diagonal := Tissue == Predicted_Tissue]
hm[, offdiag_rank := frank(fifelse(is_diagonal, Inf, -Pct), ties.method = "first"),
   by = Tissue]
hm[, show_label := !is_diagonal & Pct >= SUBTISSUE_LABEL_MIN & offdiag_rank == 1]
hm_labels <- hm[show_label == TRUE][order(-Pct)]

diag_cells <- hm[Tissue == Predicted_Tissue]
diag_cells <- merge(diag_cells, z_matrix_subtissue[, .(Tissue, tissue_correct)],
                    by.x = "Predicted_Tissue", by.y = "Tissue", all.x = TRUE)
stopifnot(!anyNA(diag_cells$tissue_correct))

plot_D_core <- ggplot(hm, aes(x = col_lab, y = row_lab, fill = Pct)) +
  geom_tile(colour = "#f0f0f0", linewidth = 0.1) +
  geom_tile(data = diag_cells[tissue_correct == TRUE],
            aes(x = col_lab, y = row_lab), fill = NA, colour = "black",
            linewidth = 0.35, inherit.aes = FALSE) +
  geom_tile(data = diag_cells[tissue_correct == FALSE],
            aes(x = col_lab, y = row_lab), fill = NA, colour = "red",
            linetype = "22", linewidth = 0.35, inherit.aes = FALSE) +
  geom_text(data = hm_labels,
            aes(label = sprintf("%.0f", Pct), colour = Pct >= 65),
            size = pt_mm(FS_DENSE_VALUE), check_overlap = TRUE, show.legend = FALSE) +
  scale_fill_gradientn(colours = GREEN_SCALE, limits = c(0, 100),
                       name = "% of pooled samples (column-normalised)",
                       guide = guide_colourbar(barwidth = unit(24, "mm"),
                                               barheight = unit(1.8, "mm"),
                                               title.position = "left",
                                               title.vjust = 1)) +
  scale_colour_manual(values = TILE_TEXT, guide = "none") +
  scale_x_discrete(expand = c(0, 0)) +
  scale_y_discrete(expand = c(0, 0)) +
  labs(x = "Predicted GTEx subtissue", y = "GTEx subtissue") +
  theme_nature_heatmap(x_angle = 45, label_size = FS_HEAT_LABEL_H) +
  theme(legend.position = "bottom", legend.direction = "horizontal")

panel_D <- add_tag(wrap_elements(full = plot_D_core), "D")

options(repr.plot.width = 7.2, repr.plot.height = 4.5)
print(panel_D)

## Assembly and export

In [ ]:
FIG_W <- 183
FIG_H <- 210
A4_W  <- 210
A4_H  <- 297
stopifnot(FIG_W <= A4_W, FIG_H <= A4_H)

# A over B, with C and D beside them: panel A is three stacked density facets and
# reads tall and narrow, while D is the widest of the four.
supp2 <- wrap_plots(
  panel_A, panel_B, panel_C, panel_D,
  design = c(area(1, 1, 1, 1),
             area(2, 1, 2, 1),
             area(1, 2, 1, 2),
             area(2, 2, 2, 2)),
  widths  = c(1, 1.6),
  heights = c(1, 1))

ggsave(snakemake@output[["pdf"]], supp2,
       width = FIG_W, height = FIG_H, units = "mm",
       device = cairo_pdf, bg = "white")
ggsave(snakemake@output[["svg"]], supp2,
       width = FIG_W, height = FIG_H, units = "mm",
       device = svglite::svglite, bg = "white")
ggsave(snakemake@output[["png"]], supp2,
       width = FIG_W, height = FIG_H, units = "mm",
       dpi = 600, device = ragg::agg_png, bg = "white")

cat(sprintf("supp2: %.0f x %.0f mm (%s A4 %.0f x %.0f mm)\n", FIG_W, FIG_H,
            if (FIG_W <= A4_W && FIG_H <= A4_H) "fits within" else "EXCEEDS", A4_W, A4_H))
cat("output directory:", dirname(snakemake@output[["pdf"]]), "\n")